# 🎓 Sistema Inteligente de Análisis del Abandono Escolar mediante Machine Learning (SIAAE-ML)

### **Politécnico José Antonio Castillo Ferreiras (Paso de Moca, Provincia Espaillat, República Dominicana)**
**Universidad Federico Henríquez y Carvajal (UFHEC)**  
**Programa:** Maestría en Inteligencia Artificial Aplicada a la Educación y la Tecnología  
**Asignatura:** Machine Learning  
**Autor:** Ing. Adonamil Cruz Evangelista  
**Períodos Analizados:** `2024-2025` y `2025-2026`  
**Versión del Cuaderno:** Copia de Trabajo Mejorada y Compatible con Google Colab (`Proyecto_ML_SIAAE_Colab.ipynb`)

---

## 📋 Notas de Integridad Científica y Alcance del Cuaderno

1. **Conservación de Fuentes Originales:** Este cuaderno es la copia de trabajo mejorada a partir de `Archivos_proyecto/Proyecto__ML_SIAAE.ipynb`, manteniendo intacto el archivo original.
2. **Datos Institucionales 100 % Verificados:** Se utilizan exclusivamente los **33 registros agregados por sección-año** y los **24 casos de abandono escolar** ($9$ en `2024-2025` y $15$ en `2025-2026`) verificados en el documento institucional oficial (`SIAAE_ML_Abandono_Verificado.docx`).
3. **Exclusión de Traslados:** Conforme al criterio institucional del centro educativo, los traslados escolares **no** se consideran abandono escolar.
4. **Diferenciación Metodológica:**
   - **Datos Verificados y Análisis Estadístico (EDA):** Tablas comparativas y visualizaciones sobre los 33 registros reales.
   - **Modelado Exploratorio de Conteos Agregados:** Comparación entre `DummyRegressor` y `PoissonRegressor` a nivel de sección.
   - **Evaluación de Viabilidad de Regresión Logística y Random Forest:** Documentación técnica y metodológica de los requerimientos de microdatos individuales para una Fase II, **sin entrenar clasificadores individuales sobre conteos agregados ni inventar registros**.

## 1. Configuración del Entorno e Importación de Bibliotecas
Este bloque configura el entorno de ejecución en **Google Colab** o entorno local de Python, asegurando compatibilidad con `pandas >= 2.0`, `numpy >= 1.24`, `matplotlib`, `seaborn` y `scikit-learn`.

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import PoissonRegressor
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')

# Configuración estética profesional para gráficos académicos
plt.rcParams['figure.figsize'] = (11, 6)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.edgecolor'] = '#cbd5e1'
plt.rcParams['axes.linewidth'] = 0.8
sns.set_theme(style='whitegrid', palette='deep')

# Crear carpetas de salida en Colab o entorno local si no existen
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('results/figures', exist_ok=True)
os.makedirs('results/reports', exist_ok=True)

print('Entorno SIAAE-ML configurado correctamente.')
print(f'Pandas: {pd.__version__} | NumPy: {np.__version__}')

## 2. Carga y Verificación Exacta de los 33 Registros Institucionales

Se construye el conjunto de datos institucional verificado correspondiente a los períodos escolares **`2024-2025`** ($17$ secciones reportadas, $9$ abandonos) y **`2025-2026`** ($16$ secciones reportadas, $15$ abandonos). La sección `1.º D` en `2025-2026` figura como **no reportada (`—`)** en el cuadro institucional verificado, por lo que no se incluye como observación numérica ni se le imputa el valor cero.

Asimismo, se homologa el nombre oficial completo de la tercera especialidad técnica del Segundo Ciclo: **«Gestión Administrativa y Tributaria»**.

In [ ]:
# Datos institucionales verificados (33 registros sección-año)
registros_verificados = [
    # Período 2024-2025 (17 secciones reportadas, 9 abandonos)
    {'anio_escolar': '2024-2025', 'ciclo': 'Primer Ciclo',  'grado': 1, 'seccion': 'A', 'especialidad': 'Nivel Secundario General (1.er Ciclo)', 'abandonos': 0},
    {'anio_escolar': '2024-2025', 'ciclo': 'Primer Ciclo',  'grado': 1, 'seccion': 'B', 'especialidad': 'Nivel Secundario General (1.er Ciclo)', 'abandonos': 0},
    {'anio_escolar': '2024-2025', 'ciclo': 'Primer Ciclo',  'grado': 1, 'seccion': 'C', 'especialidad': 'Nivel Secundario General (1.er Ciclo)', 'abandonos': 0},
    {'anio_escolar': '2024-2025', 'ciclo': 'Primer Ciclo',  'grado': 1, 'seccion': 'D', 'especialidad': 'Nivel Secundario General (1.er Ciclo)', 'abandonos': 1},
    {'anio_escolar': '2024-2025', 'ciclo': 'Primer Ciclo',  'grado': 2, 'seccion': 'A', 'especialidad': 'Nivel Secundario General (1.er Ciclo)', 'abandonos': 2},
    {'anio_escolar': '2024-2025', 'ciclo': 'Primer Ciclo',  'grado': 2, 'seccion': 'B', 'especialidad': 'Nivel Secundario General (1.er Ciclo)', 'abandonos': 2},
    {'anio_escolar': '2024-2025', 'ciclo': 'Primer Ciclo',  'grado': 3, 'seccion': 'A', 'especialidad': 'Nivel Secundario General (1.er Ciclo)', 'abandonos': 1},
    {'anio_escolar': '2024-2025', 'ciclo': 'Primer Ciclo',  'grado': 3, 'seccion': 'B', 'especialidad': 'Nivel Secundario General (1.er Ciclo)', 'abandonos': 0},
    {'anio_escolar': '2024-2025', 'ciclo': 'Segundo Ciclo', 'grado': 4, 'seccion': 'A', 'especialidad': 'Producción Agropecuaria',               'abandonos': 1},
    {'anio_escolar': '2024-2025', 'ciclo': 'Segundo Ciclo', 'grado': 4, 'seccion': 'B', 'especialidad': 'Comercio y Mercadeo',                   'abandonos': 1},
    {'anio_escolar': '2024-2025', 'ciclo': 'Segundo Ciclo', 'grado': 4, 'seccion': 'C', 'especialidad': 'Gestión Administrativa y Tributaria',   'abandonos': 0},
    {'anio_escolar': '2024-2025', 'ciclo': 'Segundo Ciclo', 'grado': 5, 'seccion': 'A', 'especialidad': 'Producción Agropecuaria',               'abandonos': 1},
    {'anio_escolar': '2024-2025', 'ciclo': 'Segundo Ciclo', 'grado': 5, 'seccion': 'B', 'especialidad': 'Comercio y Mercadeo',                   'abandonos': 0},
    {'anio_escolar': '2024-2025', 'ciclo': 'Segundo Ciclo', 'grado': 5, 'seccion': 'C', 'especialidad': 'Gestión Administrativa y Tributaria',   'abandonos': 0},
    {'anio_escolar': '2024-2025', 'ciclo': 'Segundo Ciclo', 'grado': 6, 'seccion': 'A', 'especialidad': 'Producción Agropecuaria',               'abandonos': 0},
    {'anio_escolar': '2024-2025', 'ciclo': 'Segundo Ciclo', 'grado': 6, 'seccion': 'B', 'especialidad': 'Comercio y Mercadeo',                   'abandonos': 0},
    {'anio_escolar': '2024-2025', 'ciclo': 'Segundo Ciclo', 'grado': 6, 'seccion': 'C', 'especialidad': 'Gestión Administrativa y Tributaria',   'abandonos': 0},

    # Período 2025-2026 (16 secciones reportadas, 15 abandonos; 1.º D no reportada)
    {'anio_escolar': '2025-2026', 'ciclo': 'Primer Ciclo',  'grado': 1, 'seccion': 'A', 'especialidad': 'Nivel Secundario General (1.er Ciclo)', 'abandonos': 1},
    {'anio_escolar': '2025-2026', 'ciclo': 'Primer Ciclo',  'grado': 1, 'seccion': 'B', 'especialidad': 'Nivel Secundario General (1.er Ciclo)', 'abandonos': 1},
    {'anio_escolar': '2025-2026', 'ciclo': 'Primer Ciclo',  'grado': 1, 'seccion': 'C', 'especialidad': 'Nivel Secundario General (1.er Ciclo)', 'abandonos': 4},
    {'anio_escolar': '2025-2026', 'ciclo': 'Primer Ciclo',  'grado': 2, 'seccion': 'A', 'especialidad': 'Nivel Secundario General (1.er Ciclo)', 'abandonos': 1},
    {'anio_escolar': '2025-2026', 'ciclo': 'Primer Ciclo',  'grado': 2, 'seccion': 'B', 'especialidad': 'Nivel Secundario General (1.er Ciclo)', 'abandonos': 2},
    {'anio_escolar': '2025-2026', 'ciclo': 'Primer Ciclo',  'grado': 3, 'seccion': 'A', 'especialidad': 'Nivel Secundario General (1.er Ciclo)', 'abandonos': 1},
    {'anio_escolar': '2025-2026', 'ciclo': 'Primer Ciclo',  'grado': 3, 'seccion': 'B', 'especialidad': 'Nivel Secundario General (1.er Ciclo)', 'abandonos': 0},
    {'anio_escolar': '2025-2026', 'ciclo': 'Segundo Ciclo', 'grado': 4, 'seccion': 'A', 'especialidad': 'Producción Agropecuaria',               'abandonos': 2},
    {'anio_escolar': '2025-2026', 'ciclo': 'Segundo Ciclo', 'grado': 4, 'seccion': 'B', 'especialidad': 'Comercio y Mercadeo',                   'abandonos': 0},
    {'anio_escolar': '2025-2026', 'ciclo': 'Segundo Ciclo', 'grado': 4, 'seccion': 'C', 'especialidad': 'Gestión Administrativa y Tributaria',   'abandonos': 0},
    {'anio_escolar': '2025-2026', 'ciclo': 'Segundo Ciclo', 'grado': 5, 'seccion': 'A', 'especialidad': 'Producción Agropecuaria',               'abandonos': 2},
    {'anio_escolar': '2025-2026', 'ciclo': 'Segundo Ciclo', 'grado': 5, 'seccion': 'B', 'especialidad': 'Comercio y Mercadeo',                   'abandonos': 0},
    {'anio_escolar': '2025-2026', 'ciclo': 'Segundo Ciclo', 'grado': 5, 'seccion': 'C', 'especialidad': 'Gestión Administrativa y Tributaria',   'abandonos': 1},
    {'anio_escolar': '2025-2026', 'ciclo': 'Segundo Ciclo', 'grado': 6, 'seccion': 'A', 'especialidad': 'Producción Agropecuaria',               'abandonos': 0},
    {'anio_escolar': '2025-2026', 'ciclo': 'Segundo Ciclo', 'grado': 6, 'seccion': 'B', 'especialidad': 'Comercio y Mercadeo',                   'abandonos': 0},
    {'anio_escolar': '2025-2026', 'ciclo': 'Segundo Ciclo', 'grado': 6, 'seccion': 'C', 'especialidad': 'Gestión Administrativa y Tributaria',   'abandonos': 0},
]

df = pd.DataFrame(registros_verificados)
df.insert(0, 'id_registro', range(1, len(df) + 1))

# Ingeniería de variables estructurales (ETL)
df['periodo'] = df['anio_escolar'].map({'2024-2025': 1, '2025-2026': 2})
df['grupo'] = df['grado'].astype(str) + '.º ' + df['seccion']
df['es_periodo_2'] = (df['anio_escolar'] == '2025-2026').astype(int)
df['ciclo_num'] = df['ciclo'].map({'Primer Ciclo': 1, 'Segundo Ciclo': 2})
df['es_segundo_ciclo'] = (df['ciclo_num'] == 2).astype(int)
df['grado_alto'] = (df['grado'] >= 4).astype(int)

# Verificaciones automáticas de integridad institucional
assert len(df) == 33, f'Error: se esperaban 33 registros y se encontraron {len(df)}'
assert int(df['abandonos'].sum()) == 24, f'Error: se esperaban 24 abandonos y se encontraron {df["abandonos"].sum()}'
assert int(df.loc[df['anio_escolar'] == '2024-2025', 'abandonos'].sum()) == 9
assert int(df.loc[df['anio_escolar'] == '2025-2026', 'abandonos'].sum()) == 15

# Guardar datasets verificados
df.to_csv('data/processed/abandono_agregado_siaae.csv', index=False, encoding='utf-8')

print('Verificación institucional exitosa: 33 registros sección-año | 24 abandonos totales (9 en 2024-2025 y 15 en 2025-2026).')
display(df.head(10))

## 3. Auditoría de Calidad de Datos y Estadística Descriptiva Global

In [ ]:
resumen_calidad = pd.DataFrame({
    'Tipo_Dato': df.dtypes.astype(str),
    'No_Nulos': df.notnull().sum(),
    'Nulos': df.isnull().sum(),
    'Valores_Unicos': df.nunique()
})
print('=== AUDITORÍA DE CALIDAD DE VARIABLES ===')
display(resumen_calidad)

print(f"Duplicados en clave (anio_escolar, grado, seccion): {df.duplicated(subset=['anio_escolar', 'grado', 'seccion']).sum()}")
print(f"Media global de abandonos por sección: {df['abandonos'].mean():.4f}")
print(f"Mediana global: {df['abandonos'].median():.4f} | Desviación estándar muestral: {df['abandonos'].std(ddof=1):.4f}")

## 4. Análisis Exploratorio de Datos (EDA) — Tablas Comparativas `2024-2025` vs. `2025-2026`

In [ ]:
# 4.1 Tabla comparativa por Año Escolar
tabla_periodo = df.groupby('anio_escolar', observed=True).agg(
    Secciones_Reportadas=('abandonos', 'count'),
    Total_Abandonos=('abandonos', 'sum'),
    Promedio_Seccion=('abandonos', 'mean'),
    Mediana=('abandonos', 'median'),
    Desv_Estandar=('abandonos', 'std'),
    Maximo_Seccion=('abandonos', 'max')
).round(4)

print('=== 4.1 COMPARATIVA POR PERÍODO ACADÉMICO ===')
display(tabla_periodo)

# 4.2 Tabla comparativa por Ciclo Educativo y Año Escolar
tabla_ciclo = df.pivot_table(
    index='ciclo',
    columns='anio_escolar',
    values='abandonos',
    aggfunc=['count', 'sum', 'mean'],
    observed=True
).round(4)
print('\n=== 4.2 COMPARATIVA POR CICLO EDUCATIVO ===')
display(tabla_ciclo)

# 4.3 Tabla comparativa por Grado Escolar
tabla_grado = df.pivot_table(
    index=['ciclo', 'grado'],
    columns='anio_escolar',
    values='abandonos',
    aggfunc=['count', 'sum', 'mean'],
    observed=True
).round(4)
print('\n=== 4.3 COMPARATIVA POR GRADO ESCOLAR (1.º A 6.º) ===')
display(tabla_grado)

# 4.4 Tabla comparativa por Especialidad Técnico-Profesional (Segundo Ciclo)
df_segundo_ciclo = df[df['ciclo'] == 'Segundo Ciclo']
tabla_especialidad = df_segundo_ciclo.pivot_table(
    index='especialidad',
    columns='anio_escolar',
    values='abandonos',
    aggfunc=['count', 'sum', 'mean'],
    observed=True
).round(4)
print('\n=== 4.4 COMPARATIVA POR ESPECIALIDAD TÉCNICO-PROFESIONAL (SEGUNDO CICLO) ===')
display(tabla_especialidad)

## 5. Visualizaciones Profesionales Comparativas
Se generan las visualizaciones comparativas utilizando títulos limpios compatibles con cualquier fuente tipográfica de Matplotlib en Google Colab.

In [ ]:
# Figura 1: Comparativa por Período y Ciclo Educativo
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))

res_periodo = df.groupby('anio_escolar', observed=True)['abandonos'].sum().reset_index()
bars1 = axes[0].bar(res_periodo['anio_escolar'], res_periodo['abandonos'], color=['#1e3a8a', '#2563eb'], width=0.5)
axes[0].set_title('Abandono Total por Período Escolar (n = 24)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Año Escolar')
axes[0].set_ylabel('Cantidad de Abandonos')
axes[0].set_ylim(0, 18)
for b in bars1:
    h = b.get_height()
    axes[0].text(b.get_x() + b.get_width()/2, h + 0.4, f'{int(h)} casos', ha='center', fontweight='bold')

res_ciclo = df.groupby(['ciclo', 'anio_escolar'], observed=True)['abandonos'].sum().unstack()
res_ciclo.plot(kind='bar', ax=axes[1], color=['#1e3a8a', '#0d9488'], width=0.6, rot=0)
axes[1].set_title('Abandono por Ciclo Educativo y Año Escolar', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Ciclo Educativo')
axes[1].set_ylabel('Cantidad de Abandonos')
axes[1].set_ylim(0, 12)
axes[1].legend(title='Año Escolar')

plt.suptitle('Politécnico José Antonio Castillo Ferreiras — Evolución Interanual del Abandono', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Figura 2: Abandono por Grado Escolar y Mapa de Calor Sección-Año
fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))

res_grado = df.groupby(['grado', 'anio_escolar'], observed=True)['abandonos'].sum().unstack(fill_value=0)
res_grado.plot(kind='bar', ax=axes[0], color=['#1e3a8a', '#d97706'], width=0.68, rot=0)
axes[0].set_title('Abandonos por Grado Escolar (1.º a 6.º)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Grado de Secundaria')
axes[0].set_ylabel('Cantidad de Abandonos')
axes[0].set_ylim(0, 7.5)
axes[0].legend(title='Año Escolar')

matriz_calor = df.pivot_table(index='grupo', columns='anio_escolar', values='abandonos', aggfunc='sum', observed=True)
orden_grupos = [
    '1.º A', '1.º B', '1.º C', '1.º D',
    '2.º A', '2.º B',
    '3.º A', '3.º B',
    '4.º A', '4.º B', '4.º C',
    '5.º A', '5.º B', '5.º C',
    '6.º A', '6.º B', '6.º C'
]
matriz_calor = matriz_calor.reindex(orden_grupos)
sns.heatmap(matriz_calor, annot=True, fmt='.0f', cmap='YlOrRd', cbar_kws={'label': 'Abandonos'}, ax=axes[1], linewidths=0.5)
axes[1].set_title('Mapa de Calor por Sección y Año (1.º D en blanco = No reportada)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Año Escolar')
axes[1].set_ylabel('Sección Escolar')

plt.tight_layout()
plt.show()

## 6. Modelado Exploratorio de Conteos a Nivel de Sección (`DummyRegressor` vs. `PoissonRegressor`)

> **Advertencia Metodológica Importante:** Este bloque entrena exclusivamente modelos de regresión de conteo sobre las **$n = 33$ secciones-año** para explorar la relación entre la estructura curricular (`es_periodo_2`, `ciclo_num`, `grado`, `es_segundo_ciclo`, `grado_alto`) y el número de abandonos por sección. **No** constituye un modelo de predicción de abandono individual por estudiante.

In [ ]:
features = ['es_periodo_2', 'ciclo_num', 'grado', 'es_segundo_ciclo', 'grado_alto']
X = df[features].copy()
y = df['abandonos'].copy()
groups = df['anio_escolar'].copy()

logo = LeaveOneGroupOut()
resultados_cv = []

modelos_conteo = {
    'DummyRegressor (Media)': DummyRegressor(strategy='mean'),
    'PoissonRegressor (alpha=1.0)': PoissonRegressor(alpha=1.0, max_iter=1000)
}

for nombre_modelo, modelo in modelos_conteo.items():
    maes, rmses = [], []
    for train_idx, test_idx in logo.split(X, y, groups=groups):
        modelo.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds = modelo.predict(X.iloc[test_idx])
        maes.append(mean_absolute_error(y.iloc[test_idx], preds))
        rmses.append(np.sqrt(mean_squared_error(y.iloc[test_idx], preds)))
    resultados_cv.append({
        'Modelo_Conteo_Seccion': nombre_modelo,
        'MAE_LeaveOneGroupOut': round(float(np.mean(maes)), 4),
        'RMSE_LeaveOneGroupOut': round(float(np.mean(rmses)), 4)
    })

df_resultados_cv = pd.DataFrame(resultados_cv)
print('=== EVALUACIÓN CRUZADA POR PERÍODO (LEAVE-ONE-GROUP-OUT) ===')
display(df_resultados_cv)

# Ajuste descriptivo final de PoissonRegressor sobre las 33 observaciones
poisson = PoissonRegressor(alpha=1.0, max_iter=1000)
poisson.fit(X, y)
df['pred_poisson_seccion'] = poisson.predict(X).round(4)

df_coef = pd.DataFrame({
    'Variable': ['Intercepto'] + features,
    'Coeficiente_Log': [float(poisson.intercept_)] + [float(c) for c in poisson.coef_],
    'Factor_Multiplicativo_Exp': [float(np.exp(poisson.intercept_))] + [float(np.exp(c)) for c in poisson.coef_]
}).round(6)

print('\n=== PARÁMETROS DEL MODELO EXPLORATORIO POISSON (NIVEL SECCIÓN) ===')
display(df_coef)

## 7. Evaluación de Viabilidad de Regresión Logística y Random Forest (Sin Entrenamiento Individual sobre Datos Agregados)

De acuerdo con los lineamientos metodológicos del proyecto **SIAAE-ML**, **no se entrenan modelos predictivos de abandono individual con los 33 registros agregados** por las siguientes razones científicas:

1. **Falacia ecológica:** Los conteos por sección ($y_j \in \{0, 1, 2, 4\}$) describen el comportamiento agregado del aula, no las causas ni la probabilidad individual de cada estudiante ($y_i \in \{0, 1\}$).
2. **Ausencia de denominadores de matrícula ($N_j$) y de controles individuales ($y_i = 0$):** Se conocen los $24$ casos de abandono, pero no los registros individuales de los estudiantes que permanecieron en cada sección.
3. **Prevención de sobreajuste y métricas artificiales:** Binarizar $33$ filas o simular estudiantes sintéticos violaría el principio de trabajar exclusivamente con datos institucionales comprobados.

A continuación, se documenta de forma estructurada la matriz de requerimientos de datos individuales para habilitar **Regresión Logística (Modelo A)** y **Random Forest Classifier (Modelo B)** en una Fase II del sistema.

In [ ]:
especificacion_fase2 = pd.DataFrame([
    {
        'Modelo_Propuesto': 'Modelo A: Regresión Logística Binaria',
        'Rol_en_SIAAE_ML': 'Línea base interpretable de probabilidad individual P(y_i = 1 | x_i)',
        'Unidad_Observacion_Requerida': 'Estudiante-Año Escolar (anonimizado con hash SHA-256)',
        'Variables_Adicionales_Minimas': 'Matrícula por sección, % asistencia 1.er bimestre, promedio parcial P1, asignaturas reprobadas, sobreedad, repitencia previa, alertas de orientación',
        'Estado_Actual_Bloque_A': 'Documentado (No entrenado sobre n=33 agregados por rigor metodológico)'
    },
    {
        'Modelo_Propuesto': 'Modelo B: Random Forest Classifier',
        'Rol_en_SIAAE_ML': 'Ensamble no lineal para capturar interacciones complejas (Grado x Asistencia x Sobreedad)',
        'Unidad_Observacion_Requerida': 'Estudiante-Año Escolar (anonimizado con hash SHA-256)',
        'Variables_Adicionales_Minimas': 'Mismas variables pre-abandono + historial longitudinal e importancia de variables (Gini / SHAP)',
        'Estado_Actual_Bloque_A': 'Documentado (No entrenado sobre n=33 agregados por rigor metodológico)'
    }
])

print('=== MATRIZ DE VIABILIDAD TÉCNICA PARA MODELOS SUPERVISADOS INDIVIDUALES (FASE II) ===')
display(especificacion_fase2)

## 8. Exportación del Simulador Exploratorio de Conteos por Sección (HTML Compatible con JavaScript)
Se corrige la serialización de tipos `np.float64` a `float` nativo de Python mediante `json.dumps(...)`, garantizando que el archivo interactivo `simulador_siaae.html` funcione sin errores en cualquier navegador.

In [ ]:
coef_dict = {k: round(float(v), 6) for k, v in zip(features, poisson.coef_)}
intercept_val = round(float(poisson.intercept_), 6)

html_simulador = f"""<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <title>Simulador Exploratorio de Conteo por Sección — SIAAE-ML</title>
  <style>
    body {{ font-family: system-ui, sans-serif; max-width: 680px; margin: 30px auto; padding: 20px; background: #f8fafc; color: #0f172a; }}
    .card {{ background: #fff; border: 1px solid #cbd5e1; border-radius: 12px; padding: 24px; box-shadow: 0 4px 12px rgba(0,0,0,0.05); }}
    label {{ display: block; font-weight: 600; margin-top: 12px; }}
    select, button {{ width: 100%; padding: 10px; margin-top: 6px; border-radius: 8px; border: 1px solid #94a3b8; font-size: 14px; }}
    button {{ background: #1e3a8a; color: #fff; font-weight: 700; border: none; margin-top: 18px; cursor: pointer; }}
    .result {{ margin-top: 18px; padding: 14px; border-radius: 8px; background: #eff6ff; border-left: 4px solid #2563eb; }}
  </style>
</head>
<body>
  <div class="card">
    <h2>Simulador Exploratorio Poisson (Nivel Sección)</h2>
    <p>Estimación exploratoria del conteo esperado de abandonos por sección basada en los 33 registros verificados.</p>
    <label>Período Escolar:</label>
    <select id="periodo">
      <option value="0">2024-2025</option>
      <option value="1" selected>2025-2026</option>
    </select>
    <label>Grado Escolar:</label>
    <select id="grado">
      <option value="1">1.º de Secundaria (Primer Ciclo)</option>
      <option value="2">2.º de Secundaria (Primer Ciclo)</option>
      <option value="3">3.º de Secundaria (Primer Ciclo)</option>
      <option value="4">4.º de Secundaria (Segundo Ciclo Técnico)</option>
      <option value="5">5.º de Secundaria (Segundo Ciclo Técnico)</option>
      <option value="6">6.º de Secundaria (Segundo Ciclo Técnico)</option>
    </select>
    <button onclick="calcular()">Calcular Conteo Esperado en la Sección</button>
    <div id="salida" class="result">Seleccione los parámetros y presione calcular.</div>
  </div>
  <script>
    const intercept = {intercept_val};
    const coef = {json.dumps(coef_dict)};
    function calcular() {{
      const es_periodo_2 = parseInt(document.getElementById('periodo').value);
      const grado = parseInt(document.getElementById('grado').value);
      const ciclo_num = grado >= 4 ? 2 : 1;
      const es_segundo_ciclo = grado >= 4 ? 1 : 0;
      const grado_alto = grado >= 4 ? 1 : 0;
      const eta = intercept
        + coef.es_periodo_2 * es_periodo_2
        + coef.ciclo_num * ciclo_num
        + coef.grado * grado
        + coef.es_segundo_ciclo * es_segundo_ciclo
        + coef.grado_alto * grado_alto;
      const lambda = Math.exp(eta);
      document.getElementById('salida').innerHTML =
        '<strong>Conteo esperado (λ):</strong> ' + lambda.toFixed(3) + ' abandonos por sección.';
    }}
  </script>
</body>
</html>"""

with open('results/reports/simulador_siaae.html', 'w', encoding='utf-8') as f:
    f.write(html_simulador)

print('Simulador HTML exportado correctamente en results/reports/simulador_siaae.html')